# 05 — Persistence baseline
Reproduce the frozen persistence model's 15-minute-ahead scores using the same validation/test rows as Linear Regression and Random Forest. No training, tuning or artifact writes occur. All observations are synthetic.

In [1]:
from pathlib import Path
import os, sys, json
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ml/data/phase3_dataset_manifest.json').is_file())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
approved = Path(os.environ['LOCALAPPDATA']) / 'SmartParking/ml-runtime/Scripts/python.exe'
assert Path(sys.executable).resolve() == approved.resolve()
import numpy as np
import pandas as pd
from ml.features import FEATURE_COLUMNS
from ml.phase4_models import load_predictor, metrics, predictions_agree
from ml.train_phase4 import read_partition, verify_manifest
manifest = json.loads(Path('ml/data/phase3_dataset_manifest.json').read_text(encoding='utf-8'))
results = json.loads(Path('ml/artifacts/phase4/model_results.json').read_text(encoding='utf-8'))
verify_manifest(manifest)
bundle = load_predictor(results['artifact']['path'], results['artifact']['sha256'])
print('Interpreter:', sys.executable)

Interpreter: C:\Users\vighn\AppData\Local\SmartParking\ml-runtime\Scripts\python.exe


## Model and fair comparison
ŷ(t+15)=occupied(t). The baseline has no fitted parameters or preprocessing. It is strong when stays are long relative to 15 minutes. Each partition is read using the existing frozen reader; its timestamp-row SHA256 must match the original evaluation.
Clipping to [0,12] is part of the common prediction interface and has no effect on this baseline. Metrics are computed from actual loaded-artifact predictions, not copied from the results JSON.

In [2]:
rows = []
for split in ('validation', 'test'):
    x, y, identity = read_partition(manifest, split)
    assert identity == results['partitions'][split]
    raw_prediction = bundle.models['persistence'].predict(x)
    assert np.array_equal(raw_prediction, x[:, FEATURE_COLUMNS.index('total_occupied')])
    measured = metrics(y, raw_prediction)
    frozen = results[split]['persistence']
    assert measured['rows'] == frozen['rows'] and measured['clipped_count'] == 0
    for kind in ('raw', 'clipped'):
        for key in ('mae', 'rmse', 'r2'):
            np.testing.assert_allclose(measured[kind][key], frozen[kind][key], rtol=0, atol=1e-12)
    named = pd.DataFrame(x, columns=FEATURE_COLUMNS, copy=False)
    bounded = bundle.predict(named, 'persistence')
    assert predictions_agree(bounded, raw_prediction)
    assert np.isfinite(bounded).all() and ((bounded >= 0) & (bounded <= 12)).all()
    rows.append({'split': split, **measured['clipped'], 'rows': len(y), 'origin_sha256': identity['origin_sha256']})
    if split == 'test':
        display(pd.DataFrame({'actual_t_plus_15': y[:8], 'prediction': bounded[:8], 'absolute_error': np.abs(y[:8] - bounded[:8])}))
display(pd.DataFrame(rows))
print('PASS: identical frozen rows, metric agreement, baseline formula and bounded reload interface')

,actual_t_plus_15,prediction,absolute_error
0,3.0,3.0,0.0
1,3.0,3.0,0.0
2,3.0,3.0,0.0
3,3.0,3.0,0.0
4,3.0,3.0,0.0
5,3.0,3.0,0.0
6,3.0,3.0,0.0
7,3.0,3.0,0.0


,split,mae,rmse,r2,rows,origin_sha256
0,validation,0.339983,0.684103,0.950447,10577,6c28dca1155df5f02c545ae508f03e006079183464b664...
1,test,0.307714,0.648092,0.945254,10578,79f277c74ba6bd39467ba2bf0f72e0312f6795028c6d6a...


PASS: identical frozen rows, metric agreement, baseline formula and bounded reload interface


In [3]:
record = {name: float(x[0, index]) for index, name in enumerate(FEATURE_COLUMNS)}
assert predictions_agree(bundle.predict(record), bundle.predict([record], 'persistence'))
print('Selected-model prediction from one feature record:', bundle.predict(record).tolist())
print('Horizon:', bundle.horizon_minutes, 'minutes; source:', bundle.data_source)

Selected-model prediction from one feature record: [3.0]
Horizon: 15 minutes; source: synthetic


## Interpretation and limits
Persistence was already selected by validation MAE in the frozen Phase 4 decision. These recomputations verify that saved decision; test results do not reselect the model. Persistence cannot anticipate sudden arrivals/departures and provides no uncertainty estimate. Synthetic performance does not imply real-site accuracy.
For future dashboard use, construct all 37 causal features using available current and historical measurements, then call the checksum-verified bundle's `predict` interface. Never fabricate unavailable history.